
# 📊 Industrial RAG 챗봇 구현 Tutorial

**Tutorial 목적**

산업체 현장에서는 설비 운전 기준, 품질 판정 규칙, 알람 대응 절차, 작업자 권한 범위와 같은 정보가 공개 데이터가 아니라 내부 문서에만 존재하는 경우가 많습니다.  
이러한 정보는 문서로 누적되어 있어도, 실제 업무 중에는 필요한 내용을 즉시 찾기 어렵고, 경험 많은 실무자의 기억에 의존하는 경우가 많습니다.

일반적인 생성형 LLM은 공개 데이터 중심으로 학습되므로, 기업 내부 기준이나 최신 사내 지침을 기본적으로 알지 못합니다.  
또한 그럴듯하지만 틀린 답변을 생성하는 할루시네이션 문제가 있기 때문에, 산업체 문서를 그대로 반영해야 하는 환경에서는 한계가 분명합니다.

이 튜토리얼은 산업체 내부 SOP 텍스트를 기반으로 RAG 챗봇을 구성하고, 다음 항목을 단계적으로 실습할 수 있도록 구성합니다.

- 세그먼트 식별자 기반 문서 구조화
- 청킹(Chunking) 전략 비교
- 세그먼트 식별자 오류가 검색 성능에 미치는 영향
- Top-K와 Top-P의 차이 및 실습
- Re-rank를 통한 검색 품질 개선
- 지식그래프 기반 확장
- PDF/PPT 문서로 확장하는 방향

---

## 💡 목차

> (1) 사용 데이터셋 설명  
> (2) RAG 개념 설명 및 RAG 실습  
> (3) 청킹(Chunking)  
> (4) Top-P, Top-K  
> (5) Re-rank  
> (6) 지식 그래프  
> (7) PDF/PPT 문서의 RAG화  

---

## (1) 사용 데이터셋 설명

이번 튜토리얼의 기본 데이터는 산업체 내부 운영지침서 형식으로 작성한 텍스트 파일입니다.

- 파일명: `rag_chatbot_practice_industrial_sop_v2.txt`
- 특징:
  - `SEGMENT_ID`, `SEGMENT_TYPE`, `PARENT_ID` 포함
  - 설비명, 알람코드, 임계값, 금지조치, 역할 권한 포함
  - Parent-Child 구조 반영
  - 검색 성능 비교 실습에 적합하도록 일부 실험용 세그먼트 포함

이 데이터는 일반 상식형 문서가 아니라 **사내 SOP/운전 기준서**를 가정하고 만들었기 때문에, 일반 GPT와 RAG 기반 챗봇의 차이가 분명하게 드러납니다.

예를 들어 다음 질문은 일반 GPT만으로는 산업체 내부 기준을 정확히 맞히기 어렵습니다.

- “DR-01 이슬점이 -39°C로 3분 지속되면 어떻게 해야 하나?”
- “ALM-DR-221 알람이 뜨면 가장 먼저 확인할 것은?”
- “작업자가 단독으로 수행 가능한 조치는 어디까지인가?”

---

## (2) RAG 개념 설명 및 RAG 실습

**RAG (Retrieval-Augmented Generation)** 은 외부 지식 베이스에서 관련 정보를 검색한 뒤, 그 결과를 LLM 입력에 함께 넣어 응답을 생성하는 방식입니다.

다음 3가지를 구분해야 합니다.

1. **RAG 챗봇**  
   문서를 검색하여 관련 내용을 찾아 응답합니다.  
   최신 문서나 내부 문서를 반영할 수 있고, 근거 기반 응답에 유리합니다.

2. **지식 없는 일반 프롬프트**  
   모델이 원래 학습한 일반 지식만으로 답합니다.  
   산업체 내부 기준, 특정 알람코드, 작업자 권한 규정은 모를 가능성이 큽니다.

3. **지식을 프롬프트에 직접 붙이는 방식**  
   소량 문서에는 가능하지만, 문서가 길어질수록 관리가 어렵고 길이 제한 문제도 있습니다.

산업체 환경에서 RAG가 필요한 이유는 단순합니다.  
정답이 “일반적으로”가 아니라, **문서 안에 있는 특정 기준값과 절차**이기 때문입니다.

예:
- 일반 답변: “이슬점이 상승하면 제습 상태를 확인해야 합니다.”
- RAG 답변: “DR-01 이슬점이 -39°C 이상 3분 지속 시 감속 검토, -35°C 이상이면 작업중지 판단 대상입니다.”

---

## (3) 청킹(Chunking)

청킹은 문서를 검색 가능한 단위로 나누는 과정입니다.  
이 단계가 잘못되면 정답 세그먼트가 검색되지 않거나, 기준값과 예외 규칙이 분리되어 잘못된 답변이 생성될 수 있습니다.

이번 튜토리얼에서는 다음 세 가지를 비교합니다.

### 3-1. Fixed-Length Chunking
문자를 일정 길이로 잘라 저장하는 방식입니다.

- 장점: 구현이 단순함
- 단점: 의미 단위가 깨질 수 있음
- 산업체 문서 문제:
  - 임계값과 예외 규칙이 서로 다른 chunk로 분리될 수 있음
  - 알람코드 설명과 후속 조치가 분리될 수 있음

### 3-2. Segment-Based Chunking
`SEGMENT_ID`를 기준으로 세그먼트를 하나의 chunk로 쓰는 방식입니다.

- 장점:
  - 문서 설계 의도를 그대로 보존
  - 알람, 기준값, 조치 규칙이 한 묶음으로 유지됨
- 단점:
  - 세그먼트가 너무 길면 검색 정밀도가 낮아질 수 있음

### 3-3. Parent-Child Chunking
상위 요약 세그먼트와 하위 상세 세그먼트를 함께 활용하는 방식입니다.

- Parent: 검색용 요약
- Child: 실제 기준값, 금지 조치, 단계별 절차

이 방식은 산업체 SOP처럼 구조적 문서에 특히 유리합니다.

---

## 세그먼트 식별자를 잘못 넣었을 때 문제

세그먼트 식별자는 검색 성능뿐 아니라, 추적 가능성(traceability)에도 영향을 줍니다.

### 문제 유형
- 중복 ID
- PARENT_ID 누락
- 잘못된 번호 체계
- 동일 세그먼트가 서로 다른 의미로 재사용됨

### 결과
- 검색 후 어떤 원문 세그먼트에서 가져온 내용인지 추적이 어려움
- Parent-Child 확장이 깨짐
- Re-rank 이후에도 근거 세그먼트를 잘못 연결할 수 있음
- 평가셋에서 gold segment 매칭이 불안정해짐

즉, 세그먼트 식별자는 단순 장식이 아니라 **검색/평가/설명 가능성 전체를 지탱하는 핵심 메타데이터**입니다.

---

## (4) Top-P, Top-K

Top-K와 Top-P는 같은 것이 아닙니다.  
실습에서는 반드시 구분해서 봐야 합니다.

### 4-1. Top-K
검색 단계에서 상위 몇 개 문서를 후보로 가져올지 결정합니다.

예:
- K=3: 노이즈는 적지만 정답 누락 가능성 존재
- K=5: 일반적으로 균형이 좋음
- K=8 이상: recall은 좋아질 수 있지만 노이즈 증가 가능

### 4-2. Top-P
LLM 생성 단계에서 다음 토큰을 얼마나 넓게 샘플링할지 제어합니다.

산업체 RAG에서는 보통 다음처럼 접근합니다.

- 낮은 Top-P (예: 0.1~0.3): 문서 기반 답변이 더 안정적
- 높은 Top-P (예: 0.8 이상): 일반론이 섞일 가능성 증가

즉,
- **Top-K = 검색 후보 수**
- **Top-P = 생성 다양성**

산업체 문서 기반 답변에서는 Top-P를 낮추는 것이 일반적으로 더 안전합니다.

---

## (5) Re-rank

1차 검색 결과가 반드시 최적은 아닙니다.  
예를 들어 알람코드 질문에서는 FAQ, 로그 예시, 본문 설명이 동시에 검색될 수 있습니다.

이때 Re-rank를 적용하면,
질문과 가장 직접적으로 맞는 세그먼트를 상위로 재정렬할 수 있습니다.

예:
질문: `ALM-DR-221 알람이 떴을 때 첫 번째 조치는?`

- 1차 검색 결과:
  - DR-01 일반 설명
  - ALM-DR-221 로그 예시
  - ALM-DR-221 상세 기준
- Re-rank 후 기대:
  - ALM-DR-221 상세 기준
  - ALM-DR-221 로그 예시
  - DR-01 일반 설명

즉, Re-rank는 **검색 결과 후보를 더 정밀하게 정렬하는 단계**입니다.

---

## (6) 지식 그래프

지식 그래프는 문서 간 의미 관계를 명시적으로 표현하는 방법입니다.

이번 실습에서는 다음 노드 유형을 가정합니다.

- Equipment
- Alarm
- Symptom
- Cause
- Action
- Threshold
- Role

관계 예시는 다음과 같습니다.

- `ALM-DR-221` → `OCCURS_ON` → `DR-01`
- `ALM-DR-221` → `INDICATES` → `Dew Point Rise`
- `Dew Point Rise` → `CHECK_FIRST` → `Door Open History`
- `DR-01` → `HAS_THRESHOLD` → `-39°C for 3 min`
- `Operator` → `CAN_PERFORM` → `Visual inspection`
- `Operator` → `FORBIDS_ACTION` → `Heater parameter change`

지식 그래프를 활용하면 단순 유사도 검색만으로 놓칠 수 있는 관계 기반 질의 확장이 가능해집니다.

예:
- “이 알람과 연결된 설비는?”
- “작업자가 할 수 없는 조치는?”
- “이 증상과 가장 먼저 확인할 원인은?”

---

## (7) PDF/PPT 문서의 RAG화

처음에는 txt 기반으로 실습하는 것이 가장 쉽습니다.  
그 이유는 문서 구조를 직접 설계하고, 세그먼트 식별자와 메타데이터를 명확하게 넣을 수 있기 때문입니다.

하지만 실제 산업체 환경에서는 다음 문서가 자주 등장합니다.

- PDF 운영 매뉴얼
- 품질회의 PPT
- 설비 유지보수 보고서
- 작업 표준서
- 교육자료

이 문서들을 RAG화할 때는 다음을 고려해야 합니다.

1. 텍스트 추출 정확도  
2. 표/도식/캡션의 보존 여부  
3. 페이지 번호와 원문 위치 추적  
4. 제목/소제목/표/그림 기반 구조화  
5. PDF/PPT 단락을 세그먼트 단위로 재구성하는 과정

즉, PDF/PPT는 단순 텍스트 추출보다 **구조화 전처리**가 더 중요합니다.

---

## 실습 전체 흐름 요약

1. 산업체 SOP 텍스트 로드  
2. 세그먼트 식별자 검증  
3. 청킹 방식별 인덱스 생성  
4. Dense/BM25/Hybrid 검색 실험  
5. Top-K 비교  
6. Top-P 생성 비교  
7. Re-rank 적용 전후 비교  
8. 지식 그래프 관계 추출  
9. 평가셋 기반 성능 점검  

이제 아래 코드 셀부터 순서대로 실행하면 됩니다.


In [2]:

# 기본 라이브러리
from pathlib import Path
import json
import os

BASE_DIR = Path(".").resolve()

SOP_PATH = BASE_DIR / "Data/RAG/rag_chatbot_practice_industrial_sop_v2.txt"
EVAL_PATH = BASE_DIR / "Data/RAG/rag_eval_dataset.json"
KG_SCHEMA_PATH = BASE_DIR / "Data/RAG/kg_extraction_format.json"

print("BASE_DIR:", BASE_DIR)
print("SOP exists:", SOP_PATH.exists(), SOP_PATH)
print("EVAL exists:", EVAL_PATH.exists(), EVAL_PATH)
print("KG schema exists:", KG_SCHEMA_PATH.exists(), KG_SCHEMA_PATH)


BASE_DIR: /workspaces/2026_KAIST_AgenticAI_Workshop_iAILab
SOP exists: True /workspaces/2026_KAIST_AgenticAI_Workshop_iAILab/Data/RAG/rag_chatbot_practice_industrial_sop_v2.txt
EVAL exists: True /workspaces/2026_KAIST_AgenticAI_Workshop_iAILab/Data/RAG/rag_eval_dataset.json
KG schema exists: True /workspaces/2026_KAIST_AgenticAI_Workshop_iAILab/Data/RAG/kg_extraction_format.json



## 실습 환경 준비

아래 셀은 실습에 필요한 패키지를 설치하는 예시입니다.  
환경에 따라 이미 설치되어 있다면 생략해도 됩니다.

- `sentence-transformers`
- `scikit-learn`
- `rank-bm25`
- `pandas`
- `numpy`

CrossEncoder 기반 re-rank까지 실습하려면 `sentence-transformers`가 필요합니다.


In [3]:

# 필요 시 실행
# !pip install sentence-transformers scikit-learn rank-bm25 pandas numpy



## (1) 사용 데이터셋 확인

먼저 산업체 SOP 텍스트 문서 앞부분을 확인합니다.  
세그먼트 식별자, 유형, 부모 관계가 들어 있는지 점검합니다.


In [4]:

from itertools import islice

with open(SOP_PATH, "r", encoding="utf-8") as f:
    for line in islice(f, 80):
        print(line.rstrip())


문서명: PKM-PLANT-OPS-017
문서제목: 전극 코팅 라인 이상 징후 대응 및 설비 점검 내부 표준서
문서버전: Rev.2.0
문서등급: 사내용 / 교육용 샘플
작성부서: 제조기술팀 / 설비신뢰성파트
적용라인: COT-LINE-02, COT-LINE-03
관련시스템: MES, Historian, Vision Inspector, Dry Room Monitor
작성목적:
이 문서는 전극 코팅 라인에서 반복적으로 발생하는 이상 징후에 대해
현장 작업자, 공정기술자, 설비엔지니어가 동일한 기준으로 대응하도록 하기 위한 내부 표준서이다.
일반적인 설비 상식이 아니라 당사 라인 기준의 임계값, 우선조치 순서, 기록 규칙을 포함한다.

실습 안내:
1) 각 세그먼트는 고유 식별자를 가진다.
2) 세그먼트 단위 검색, 문단 청킹, 부모-자식 청킹 실습을 고려하여 작성되었다.
3) 일부 용어는 현장 별칭과 공식 명칭을 함께 사용한다.
4) 질의응답 실습 시 "일반적인 설비 상식"보다 "문서 내 기준값"을 우선한다.

SEGMENT_ID: RAG-SOP-000
SEGMENT_TYPE: DOC_META
PARENT_ID: NONE
TITLE: 문서 사용 원칙
KEYWORDS: 문서규칙, 응답원칙, 내부기준, 우선순위
BODY:
이 문서를 참조하는 챗봇 또는 검색 시스템은 외부 일반 지식보다 본 문서의 수치 기준과
현장 절차를 우선 답변해야 한다.
예를 들어 건조실 이슬점, 슬러리 점도, 코팅 두께 편차에 관해서는 업계 평균이나
교과서적 표현보다 본 문서의 허용범위와 단계별 조치를 우선 적용한다.
문서 내에서 "즉시 정지"와 "감속 운전"은 서로 다른 대응이므로 혼동하지 않는다.
또한 "작업중지"는 안전 및 품질 격리까지 포함하는 표현이고,
"라인정지"는 설비 구동 정지 자체를 의미한다.
END_SEGMENT

SEGMENT_ID: RAG-SOP-001
SEGMENT_TYPE: PARENT
PARENT_ID: NONE
TITLE: 적용 범위 및 설비 명칭 체계
KEYWOR


## (2) RAG 개념 및 Baseline 실습

가장 먼저 비교할 것은 다음 세 가지입니다.

1. 일반 GPT 식 응답 가정  
2. 문서를 단순 프롬프트에 붙이는 방식  
3. 검색 후 관련 세그먼트만 붙이는 RAG 방식  

여기서는 우선 문서를 검색 가능한 구조로 파싱합니다.


In [5]:

import re
from typing import List, Dict

def parse_sop_segments(text: str) -> List[Dict]:
    segments = []
    current = None
    body_lines = []
    in_body = False

    for raw_line in text.splitlines():
        line = raw_line.rstrip("\n")
        stripped = line.strip()

        # 빈 줄 처리
        if not stripped:
            if current is not None and in_body:
                body_lines.append("")
            continue

        # 구분선은 무시
        if stripped.startswith("===="):
            continue

        # 새 세그먼트 시작
        if stripped.startswith("SEGMENT_ID:"):
            # 혹시 이전 세그먼트가 END_SEGMENT 없이 끝났으면 저장
            if current is not None:
                current["body"] = "\n".join(body_lines).strip()
                segments.append(current)

            current = {
                "segment_id": stripped.split(":", 1)[1].strip(),
                "segment_type": None,
                "parent_id": None,
                "title": None,
                "keywords": [],
                "body": "",
            }
            body_lines = []
            in_body = False
            continue

        # 세그먼트 시작 전 문서 헤더는 무시
        if current is None:
            continue

        # 세그먼트 종료
        if stripped == "END_SEGMENT":
            current["body"] = "\n".join(body_lines).strip()
            segments.append(current)
            current = None
            body_lines = []
            in_body = False
            continue

        # 메타데이터 파싱
        if stripped.startswith("SEGMENT_TYPE:"):
            current["segment_type"] = stripped.split(":", 1)[1].strip()

        elif stripped.startswith("PARENT_ID:"):
            value = stripped.split(":", 1)[1].strip()
            current["parent_id"] = None if value in ("", "NONE", "None", "null") else value

        elif stripped.startswith("TITLE:"):
            current["title"] = stripped.split(":", 1)[1].strip()

        elif stripped.startswith("KEYWORDS:"):
            value = stripped.split(":", 1)[1].strip()
            current["keywords"] = [x.strip() for x in value.split(",") if x.strip()]

        elif stripped.startswith("BODY:"):
            in_body = True
            value = stripped.split(":", 1)[1].strip()
            if value:
                body_lines.append(value)

        else:
            if in_body:
                body_lines.append(stripped)

    # 파일 끝 처리
    if current is not None:
        current["body"] = "\n".join(body_lines).strip()
        segments.append(current)

    return segments

In [6]:
text = SOP_PATH.read_text(encoding="utf-8")
segments = parse_sop_segments(text)

print("총 세그먼트 수:", len(segments))
segments[:2]


총 세그먼트 수: 71


[{'segment_id': 'RAG-SOP-000',
  'segment_type': 'DOC_META',
  'parent_id': None,
  'title': '문서 사용 원칙',
  'keywords': ['문서규칙', '응답원칙', '내부기준', '우선순위'],
  'body': '이 문서를 참조하는 챗봇 또는 검색 시스템은 외부 일반 지식보다 본 문서의 수치 기준과\n현장 절차를 우선 답변해야 한다.\n예를 들어 건조실 이슬점, 슬러리 점도, 코팅 두께 편차에 관해서는 업계 평균이나\n교과서적 표현보다 본 문서의 허용범위와 단계별 조치를 우선 적용한다.\n문서 내에서 "즉시 정지"와 "감속 운전"은 서로 다른 대응이므로 혼동하지 않는다.\n또한 "작업중지"는 안전 및 품질 격리까지 포함하는 표현이고,\n"라인정지"는 설비 구동 정지 자체를 의미한다.'},
 {'segment_id': 'RAG-SOP-001',
  'segment_type': 'PARENT',
  'parent_id': None,
  'title': '적용 범위 및 설비 명칭 체계',
  'keywords': ['라인구성', '설비명', '별칭', '적용범위'],
  'body': '본 표준서는 COT-LINE-02 및 COT-LINE-03의 슬러리 공급부, 코팅 헤드, 건조부,\n장력 제어부, 비전 검사부를 포함한다.\n현장에서는 코팅 헤드를 "헤드", 건조부를 "오븐", Dry Room 연결구역을 "드라이존"이라 부르지만,\n기록지에는 공식 명칭을 사용해야 한다.\n설비명 표기 예시는 다음과 같다.\n- SL-02: Slurry Day Tank 02\n- PH-02: Feed Pump 02\n- CH-02: Coating Head 02\n- OZ-1 ~ OZ-5: Oven Zone 1~5\n- DR-01: Dry Room Interface 01\n- VI-02: Vision Inspector 02\n동일 알람이라도 라인과 설비 위치가 다르면 원인이 달라질 수 있으므로,\n답변 시


## (3) 청킹(Chunking)

같은 문서라도 청킹 방식에 따라 검색 결과가 달라집니다.  
아래 셀에서는 세 가지 청킹을 생성합니다.

- fixed
- segment
- parent_child


In [9]:

def build_fixed_chunks(text: str, chunk_size: int = 500, overlap: int = 50):
    chunks = []
    start = 0
    idx = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append({
            "chunk_id": f"fixed-{idx}",
            "text": text[start:end]
        })
        idx += 1
        if end == len(text):
            break
        start = end - overlap
    return chunks

def build_segment_chunks(segments):
    chunks = []
    for s in segments:
        chunks.append({
            "chunk_id": s["segment_id"],
            "text": f'{s["title"]}\n' + s["body"],
            "meta": s
        })
    return chunks

def build_parent_child_chunks(segments):
    by_id = {s["segment_id"]: s for s in segments if s["segment_id"]}
    chunks = []
    for s in segments:
        parent_text = ""
        if s["parent_id"] and s["parent_id"] in by_id:
            parent = by_id[s["parent_id"]]
            parent_text = f'[PARENT] {parent["title"]}\n{parent["body"]}\n\n'
        chunks.append({
            "chunk_id": s["segment_id"],
            "text": parent_text + f'[CHILD] {s["title"]}\n{s["body"]}',
            "meta": s
        })
    return chunks

fixed_chunks = build_fixed_chunks(text)
segment_chunks = build_segment_chunks(segments)
parent_child_chunks = build_parent_child_chunks(segments)

print("fixed:", len(fixed_chunks))
print("segment:", len(segment_chunks))
print("parent_child:", len(parent_child_chunks))
print(parent_child_chunks[0]["text"][:500])


fixed: 84
segment: 71
parent_child: 71
[CHILD] 문서 사용 원칙
이 문서를 참조하는 챗봇 또는 검색 시스템은 외부 일반 지식보다 본 문서의 수치 기준과
현장 절차를 우선 답변해야 한다.
예를 들어 건조실 이슬점, 슬러리 점도, 코팅 두께 편차에 관해서는 업계 평균이나
교과서적 표현보다 본 문서의 허용범위와 단계별 조치를 우선 적용한다.
문서 내에서 "즉시 정지"와 "감속 운전"은 서로 다른 대응이므로 혼동하지 않는다.
또한 "작업중지"는 안전 및 품질 격리까지 포함하는 표현이고,
"라인정지"는 설비 구동 정지 자체를 의미한다.



## 세그먼트 식별자 오류 실습

이번 단계는 일부러 문서를 망가뜨려서 검색 성능 차이를 보는 실험입니다.

다음 문제를 주입합니다.

- duplicate_id
- missing_parent
- irregular_id

이 실습은 **세그먼트 식별자가 검색 품질과 추적성에 왜 중요한지** 확인하는 데 목적이 있습니다.


In [8]:

import copy

def corrupt_duplicate_id(segments):
    corrupted = copy.deepcopy(segments)
    if len(corrupted) >= 2:
        corrupted[1]["segment_id"] = corrupted[0]["segment_id"]
    return corrupted

def corrupt_missing_parent(segments):
    corrupted = copy.deepcopy(segments)
    for s in corrupted:
        if s["parent_id"] is not None:
            s["parent_id"] = "BROKEN-PARENT-ID"
            break
    return corrupted

def corrupt_irregular_id(segments):
    corrupted = copy.deepcopy(segments)
    for i, s in enumerate(corrupted[:5]):
        s["segment_id"] = f"bad id {i}"
    return corrupted

segments_dup = corrupt_duplicate_id(segments)
segments_missing_parent = corrupt_missing_parent(segments)
segments_irregular = corrupt_irregular_id(segments)

print("정상 예시:", segments[0]["segment_id"], segments[1]["segment_id"])
print("중복 식별자 예시:", segments_dup[0]["segment_id"], segments_dup[1]["segment_id"])
print("상위 세그먼트 연결 오류 예시:", next((s for s in segments_missing_parent if s["parent_id"] == "BROKEN-PARENT-ID"), None))
print("비정형 식별자 예시:", [s["segment_id"] for s in segments_irregular[:5]])

정상 예시: RAG-SOP-000 RAG-SOP-001
중복 식별자 예시: RAG-SOP-000 RAG-SOP-000
상위 세그먼트 연결 오류 예시: {'segment_id': 'RAG-SOP-002', 'segment_type': 'CHILD', 'parent_id': 'BROKEN-PARENT-ID', 'title': '공식 명칭과 현장 별칭 매핑', 'keywords': ['별칭', '용어사전', '현장용어', '매핑'], 'body': '현장 질의에서 자주 혼용되는 표현은 아래와 같이 해석한다.\n- "슬러리탱크" = SL-02 또는 SL-03 Day Tank\n- "정량펌프" = PH-02 Feed Pump\n- "헤드 떨림" = CH-02 Coating Head Vibration 또는 Mount Looseness\n- "오븐 3번" = OZ-3\n- "드라이룸 연결부" = DR-01\n- "검사기 카메라" = VI-02 Camera Module\n질의에 별칭만 있는 경우에도 답변에는 공식 설비명을 병기한다.\n예: "오븐 3번 온도 흔들림" → "OZ-3 온도 편차"로 정규화'}
비정형 식별자 예시: ['bad id 0', 'bad id 1', 'bad id 2', 'bad id 3', 'bad id 4']



## (4) Top-K 실습

검색에서는 보통 상위 K개 chunk를 가져온 뒤 그 안에서 답을 생성합니다.  
여기서는 간단한 TF-IDF 기반 baseline 검색부터 실습합니다.


In [10]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class SimpleRetriever:
    def __init__(self, chunks):
        self.chunks = chunks
        self.texts = [c["text"] for c in chunks]
        self.vectorizer = TfidfVectorizer()
        self.matrix = self.vectorizer.fit_transform(self.texts)

    def search(self, query, top_k=5):
        q = self.vectorizer.transform([query])
        scores = cosine_similarity(q, self.matrix)[0]
        ranked = sorted(
            [{"score": float(scores[i]), **self.chunks[i]} for i in range(len(self.chunks))],
            key=lambda x: x["score"],
            reverse=True
        )
        return ranked[:top_k]

retriever = SimpleRetriever(segment_chunks)

query = "ALM-DR-221 알람이 떴을 때 가장 먼저 확인할 것은?"
results = retriever.search(query, top_k=5)

for r in results:
    print("=" * 80)
    print("chunk_id:", r["chunk_id"], "score:", round(r["score"], 4))
    print(r["text"][:400])


chunk_id: RAG-SOP-032 score: 0.1908
ALM-DR-221 해석
ALM-DR-221은 DR-01 이슬점 상승 경보다.
-39°C 이상 3분 지속 여부를 함께 봐야 하며,
도어 개방 이력과 제습기 재생 상태를 우선 확인한다.
-35°C 이상이면 작업중지 판단 대상이다.
chunk_id: RAG-SOP-063 score: 0.1607
Re-rank 전후 기대 결과 예시
예시 질의:
"ALM-DR-221 뜨면 제일 먼저 뭐 확인해?"

re-rank 전 예상 상위 후보 예:
1. RAG-SOP-006 건조부 이슬점 기준 해석 규칙
2. RAG-SOP-041 기록 예시 1 - 이슬점 상승
3. RAG-SOP-032 ALM-DR-221 해석
4. RAG-SOP-020 DR-01 이슬점 상승 단계별 조치
5. RAG-SOP-040 FAQ

re-rank 후 기대 상위 후보 예:
1. RAG-SOP-032 ALM-DR-221 해석
2. RAG-SOP-006 건조부 이슬점 기준 해석 규칙
3. RAG-SOP-021 도어 개방 이력 해석 주의사항
4. RAG-SOP-020 DR-01 이슬점 상승 단계별 조치

핵심:
직접적인 질문-응답 관계를 가진 
chunk_id: RAG-SOP-006 score: 0.1577
건조부 이슬점 기준 해석 규칙
DR-01의 기준은 -42°C 이하 유지이다.
-40°C는 업계 평균으로는 양호하게 볼 수 있으나, 당사 라인에서는 경고 구간이다.
-39°C 이상이 3분 이상 지속되면 감속 운전을 검토한다.
-37°C 이상이 2분 이상 지속되면 신규 투입을 중단하고 품질팀에 예비 통보한다.
-35°C 이상이면 즉시 작업중지 판단 대상이다.
단, 센서 교정 직후 1회성 튐값은 30초 확인 후 판단한다.
이슬점 이상 시 가장 먼저 확인할 항목은 DR-01 도어 개방 이력, 제습기 재생상태,
외기 유입 댐퍼 상태 순서이다.
chunk_id: RAG-SOP-065 score: 0.1348
지식그래프 노드 유형 정의
권장 노드 유형은 


### Top-K 비교

같은 질의에 대해 Top-K를 바꾸면 어떤 변화가 생기는지 확인합니다.


In [11]:

query = "DR-01 이슬점이 -39°C로 3분 지속되면 어떻게 해야 하나?"

for k in [3, 5, 8]:
    print(f"\n===== TOP-K = {k} =====")
    results = retriever.search(query, top_k=k)
    for r in results:
        print(r["chunk_id"], round(r["score"], 4), "|", r["text"][:120].replace("\n", " "))



===== TOP-K = 3 =====
RAG-SOP-006 0.1947 | 건조부 이슬점 기준 해석 규칙 DR-01의 기준은 -42°C 이하 유지이다. -40°C는 업계 평균으로는 양호하게 볼 수 있으나, 당사 라인에서는 경고 구간이다. -39°C 이상이 3분 이상 지속되면 감속 운전을 검
RAG-SOP-020 0.1908 | DR-01 이슬점 상승 단계별 조치 DR-01 이슬점이 -42°C 초과 ~ -39°C 미만이면 경향 감시, -39°C 이상 3분 지속이면 속도 15% 감속 검토, -37°C 이상 2분 지속이면 신규 롤 투입 중단 및
RAG-SOP-032 0.1871 | ALM-DR-221 해석 ALM-DR-221은 DR-01 이슬점 상승 경보다. -39°C 이상 3분 지속 여부를 함께 봐야 하며, 도어 개방 이력과 제습기 재생 상태를 우선 확인한다. -35°C 이상이면 작업중지 판

===== TOP-K = 5 =====
RAG-SOP-006 0.1947 | 건조부 이슬점 기준 해석 규칙 DR-01의 기준은 -42°C 이하 유지이다. -40°C는 업계 평균으로는 양호하게 볼 수 있으나, 당사 라인에서는 경고 구간이다. -39°C 이상이 3분 이상 지속되면 감속 운전을 검
RAG-SOP-020 0.1908 | DR-01 이슬점 상승 단계별 조치 DR-01 이슬점이 -42°C 초과 ~ -39°C 미만이면 경향 감시, -39°C 이상 3분 지속이면 속도 15% 감속 검토, -37°C 이상 2분 지속이면 신규 롤 투입 중단 및
RAG-SOP-032 0.1871 | ALM-DR-221 해석 ALM-DR-221은 DR-01 이슬점 상승 경보다. -39°C 이상 3분 지속 여부를 함께 봐야 하며, 도어 개방 이력과 제습기 재생 상태를 우선 확인한다. -35°C 이상이면 작업중지 판
RAG-SOP-040 0.1417 | 자주 묻는 현장 질문 Q1. DR-01 이슬점이 -40°C면 괜찮은가? A1. 업계 일반론으로는 양호하게 볼 수 있으나, 당사 기준에서는 경고 구


## Top-P 실습 설명

Top-P는 검색이 아니라 **생성 단계 샘플링 설정**입니다.  
즉, retrieval 결과를 동일하게 유지한 상태에서 LLM 파라미터만 바꿔야 비교가 됩니다.

산업체 RAG에서는 보통 Top-P를 낮게 두는 편이 더 안전합니다.


In [12]:

retrieved_context = "\n\n".join([r["text"] for r in retriever.search(query, top_k=5)])

prompt_pack = {
    "question": query,
    "retrieved_context": retrieved_context,
    "generation_settings": [
        {"name": "stable", "temperature": 0.1, "top_p": 0.2},
        {"name": "balanced", "temperature": 0.3, "top_p": 0.5},
        {"name": "creative_risk", "temperature": 0.7, "top_p": 0.9},
    ]
}

print(json.dumps(prompt_pack, ensure_ascii=False, indent=2)[:2500])


{
  "question": "DR-01 이슬점이 -39°C로 3분 지속되면 어떻게 해야 하나?",
  "retrieved_context": "건조부 이슬점 기준 해석 규칙\nDR-01의 기준은 -42°C 이하 유지이다.\n-40°C는 업계 평균으로는 양호하게 볼 수 있으나, 당사 라인에서는 경고 구간이다.\n-39°C 이상이 3분 이상 지속되면 감속 운전을 검토한다.\n-37°C 이상이 2분 이상 지속되면 신규 투입을 중단하고 품질팀에 예비 통보한다.\n-35°C 이상이면 즉시 작업중지 판단 대상이다.\n단, 센서 교정 직후 1회성 튐값은 30초 확인 후 판단한다.\n이슬점 이상 시 가장 먼저 확인할 항목은 DR-01 도어 개방 이력, 제습기 재생상태,\n외기 유입 댐퍼 상태 순서이다.\n\nDR-01 이슬점 상승 단계별 조치\nDR-01 이슬점이\n-42°C 초과 ~ -39°C 미만이면 경향 감시,\n-39°C 이상 3분 지속이면 속도 15% 감속 검토,\n-37°C 이상 2분 지속이면 신규 롤 투입 중단 및 품질 예비통보,\n-35°C 이상이면 즉시 작업중지 판단 대상이다.\n여기서 \"즉시 작업중지\"는 이미 진행 중인 웹을 무조건 절단하라는 뜻이 아니라,\n신규 투입을 멈추고 안전/품질 담당자 승인 전 재가동하지 않는 것을 뜻한다.\n\nALM-DR-221 해석\nALM-DR-221은 DR-01 이슬점 상승 경보다.\n-39°C 이상 3분 지속 여부를 함께 봐야 하며,\n도어 개방 이력과 제습기 재생 상태를 우선 확인한다.\n-35°C 이상이면 작업중지 판단 대상이다.\n\n자주 묻는 현장 질문\nQ1. DR-01 이슬점이 -40°C면 괜찮은가?\nA1. 업계 일반론으로는 양호하게 볼 수 있으나, 당사 기준에서는 경고 구간이다.\n-39°C 이상 3분 지속 여부를 보고 감속 검토가 필요하다.\n\nQ2. 비전 불량률이 3% 넘으면 바로 불량인가?\nA2. 아니다. 렌즈/조명/실물 샘플 3매를 먼저 확인해야 한다.\n비전NG와 실물NG는 분리 기록한다.


## (5) Re-rank 실습

1차 검색 결과를 다시 정렬하는 간단한 re-rank 예시입니다.  
여기서는 CrossEncoder 대신 휴리스틱 기반 점수 보정을 먼저 보여줍니다.

- 알람코드 직접 일치
- 숫자/임계값 포함 여부
- 제목 일치 가중치


In [13]:

import re

def heuristic_rerank(query, candidates):
    query_upper = query.upper()
    scored = []

    for c in candidates:
        bonus = 0.0
        text_upper = c["text"].upper()

        # 알람코드 직접 매칭 가중치
        alarm_codes = re.findall(r"ALM-[A-Z]+-\d+", query_upper)
        for code in alarm_codes:
            if code in text_upper:
                bonus += 0.5

        # 수치/임계값 언급 가중치
        if any(x in text_upper for x in ["-39", "-35", "3 MIN", "3분", "2분"]):
            bonus += 0.2

        # 제목 일치 가중치
        title = c.get("meta", {}).get("title", "") if "meta" in c else ""
        if title and any(tok in title.upper() for tok in query_upper.split()):
            bonus += 0.1

        new_score = c["score"] + bonus
        scored.append({**c, "rerank_score": round(new_score, 4), "bonus": round(bonus, 4)})

    return sorted(scored, key=lambda x: x["rerank_score"], reverse=True)

query = "ALM-DR-221 알람이 떴을 때 가장 먼저 확인할 것은?"
initial = retriever.search(query, top_k=8)
reranked = heuristic_rerank(query, initial)

print("=== Before Re-rank ===")
for r in initial[:5]:
    print(r["chunk_id"], round(r["score"], 4))

print("\n=== After Re-rank ===")
for r in reranked[:5]:
    print(r["chunk_id"], "base:", round(r["score"], 4), "bonus:", r["bonus"], "final:", r["rerank_score"])


=== Before Re-rank ===
RAG-SOP-032 0.1908
RAG-SOP-063 0.1607
RAG-SOP-006 0.1577
RAG-SOP-065 0.1348
RAG-SOP-034 0.1294

=== After Re-rank ===
RAG-SOP-032 base: 0.1908 bonus: 0.8 final: 0.9908
RAG-SOP-066 base: 0.1157 bonus: 0.7 final: 0.8157
RAG-SOP-063 base: 0.1607 bonus: 0.5 final: 0.6607
RAG-SOP-065 base: 0.1348 bonus: 0.5 final: 0.6348
RAG-SOP-061 base: 0.126 bonus: 0.5 final: 0.626



## (6) 지식 그래프

지식 그래프는 문서 속 객체와 관계를 구조적으로 뽑아내는 방법입니다.  
이번 튜토리얼에서는 엔티티/관계 추출 포맷을 이용해 간단한 그래프 후보를 만듭니다.


In [14]:

kg_schema = json.loads(KG_SCHEMA_PATH.read_text(encoding="utf-8"))
print(json.dumps(kg_schema, ensure_ascii=False, indent=2)[:3000])


{
  "schema_version": "1.0",
  "document_id": "PKM-PLANT-OPS-017",
  "document_title": "전극 코팅 라인 이상 징후 대응 및 설비 점검 내부 표준서",
  "purpose": "산업체 SOP 기반 RAG에서 Graph RAG 확장용 엔티티/관계 추출 포맷",
  "entity_types": [
    {
      "type": "Equipment",
      "required_fields": [
        "entity_id",
        "canonical_name"
      ],
      "optional_fields": [
        "aliases",
        "line_scope",
        "description"
      ],
      "examples": [
        {
          "entity_id": "EQ-DR-01",
          "canonical_name": "DR-01",
          "aliases": [
            "드라이룸 연결부",
            "드라이존"
          ]
        },
        {
          "entity_id": "EQ-OZ-3",
          "canonical_name": "OZ-3",
          "aliases": [
            "오븐 3번"
          ]
        },
        {
          "entity_id": "EQ-CH-02",
          "canonical_name": "CH-02",
          "aliases": [
            "헤드",
            "코팅 헤드"
          ]
        }
      ]
    },
    {
      "type": "Alarm",
      "required_fields": [
        "e

In [15]:

def extract_graph_candidates(segments):
    triples = []

    for s in segments:
        sid = s["segment_id"]
        text = f'{s["title"]}\n{s["body"]}'

        # 알람코드 추출
        for alarm in re.findall(r"ALM-[A-Z]+-\d+", text):
            triples.append({
                "source": alarm,
                "relation": "MENTIONED_IN",
                "target": sid,
                "evidence_segment_id": sid
            })

        # 설비명 추출 예시
        for equip in re.findall(r"\b[A-Z]{2}-\d{2}\b", text):
            triples.append({
                "source": sid,
                "relation": "RELATED_TO",
                "target": equip,
                "evidence_segment_id": sid
            })

        # 임계값 추출 예시
        for threshold in re.findall(r"-\d+\s?°C", text):
            triples.append({
                "source": sid,
                "relation": "HAS_THRESHOLD",
                "target": threshold,
                "evidence_segment_id": sid
            })

    return triples

triples = extract_graph_candidates(segments)
print("추출 triple 수:", len(triples))
triples[:15]


추출 triple 수: 105


[{'source': 'RAG-SOP-001',
  'relation': 'RELATED_TO',
  'target': 'SL-02',
  'evidence_segment_id': 'RAG-SOP-001'},
 {'source': 'RAG-SOP-001',
  'relation': 'RELATED_TO',
  'target': 'PH-02',
  'evidence_segment_id': 'RAG-SOP-001'},
 {'source': 'RAG-SOP-001',
  'relation': 'RELATED_TO',
  'target': 'CH-02',
  'evidence_segment_id': 'RAG-SOP-001'},
 {'source': 'RAG-SOP-001',
  'relation': 'RELATED_TO',
  'target': 'DR-01',
  'evidence_segment_id': 'RAG-SOP-001'},
 {'source': 'RAG-SOP-001',
  'relation': 'RELATED_TO',
  'target': 'VI-02',
  'evidence_segment_id': 'RAG-SOP-001'},
 {'source': 'RAG-SOP-002',
  'relation': 'RELATED_TO',
  'target': 'SL-02',
  'evidence_segment_id': 'RAG-SOP-002'},
 {'source': 'RAG-SOP-002',
  'relation': 'RELATED_TO',
  'target': 'SL-03',
  'evidence_segment_id': 'RAG-SOP-002'},
 {'source': 'RAG-SOP-002',
  'relation': 'RELATED_TO',
  'target': 'PH-02',
  'evidence_segment_id': 'RAG-SOP-002'},
 {'source': 'RAG-SOP-002',
  'relation': 'RELATED_TO',
  'target


## 평가용 질의-정답셋 실습

평가셋에는 다음 정보가 들어 있습니다.

- 질문
- 정답 세그먼트 ID
- 반드시 들어가야 할 표현
- 들어가면 안 되는 표현

이를 이용해 Hit@K나 간단한 정확도 점검을 수행할 수 있습니다.


In [16]:

eval_data = json.loads(EVAL_PATH.read_text(encoding="utf-8"))
print("평가 문항 수:", len(eval_data))
eval_data[:2]


평가 문항 수: 20


[{'query_id': 'Q-01',
  'question': 'DR-01 이슬점이 -40°C인데 계속 돌려도 되나?',
  'gold_segment_ids': ['RAG-SOP-006', 'RAG-SOP-020', 'RAG-SOP-032'],
  'answer_must_include': ['-42°C 이하 유지',
   '-40°C는 당사 기준 경고 구간',
   '-39°C 이상 3분 지속 시 감속 검토'],
  'answer_must_not_include': ['일반적으로는 괜찮다'],
  'query_type': 'threshold_decision',
  'recommended_chunking': 'parent_child',
  'recommended_top_k': 5},
 {'query_id': 'Q-02',
  'question': 'ALM-DR-221 떴는데 제일 먼저 뭘 확인해야 해?',
  'gold_segment_ids': ['RAG-SOP-032', 'RAG-SOP-006', 'RAG-SOP-021'],
  'answer_must_include': ['도어 개방 이력', '제습기 재생 상태', 'DR-01'],
  'answer_must_not_include': ['바로 정상 복귀 가능'],
  'query_type': 'alarm_first_check',
  'recommended_chunking': 'segment',
  'recommended_top_k': 8}]

In [17]:

def evaluate_hit_at_k(retriever, eval_data, k=5):
    hits = 0
    total = len(eval_data)

    for item in eval_data:
        results = retriever.search(item["question"], top_k=k)
        retrieved_ids = [r["chunk_id"] for r in results]
        gold_ids = item["gold_segment_ids"]
        if any(g in retrieved_ids for g in gold_ids):
            hits += 1

    return hits / total if total else 0.0

for k in [1, 3, 5, 8]:
    score = evaluate_hit_at_k(retriever, eval_data, k=k)
    print(f"Hit@{k}: {score:.3f}")


Hit@1: 0.500
Hit@3: 0.700
Hit@5: 0.800
Hit@8: 0.850



## (7) PDF/PPT 문서의 RAG화 방향

txt 문서 실습이 끝났다면, 다음 확장을 고려할 수 있습니다.

### PDF
- 페이지 단위 텍스트 추출
- 제목/표/그림 캡션 구조화
- 페이지 번호 메타데이터 유지
- 표를 별도 세그먼트로 저장

### PPT
- 슬라이드 번호 유지
- 제목 / 본문 / 노트 / 도형 텍스트 분리
- 슬라이드별 요약 세그먼트 생성
- 표와 도식 설명을 별도 세그먼트화

핵심은 “그냥 텍스트 추출”이 아니라,  
**원문 구조를 최대한 보존한 세그먼트 재구성**입니다.



## 마무리

이 튜토리얼에서는 산업체 내부 SOP 문서를 대상으로 다음을 모두 연결해 보았습니다.

- RAG 기본 개념
- 세그먼트 식별자 설계
- 청킹 전략
- Top-K / Top-P
- Re-rank
- 지식 그래프
- 평가셋 기반 검증
- PDF/PPT 확장 방향

즉, 이번 자료는 “RAG 개념 설명”이 아니라,  
**산업체 실무 문서를 대상으로 검색 성능과 근거 추적성을 함께 보는 실습형 튜토리얼**입니다.
